# Phutball self-play board ladder (A100)

Runs `expert/selfplay_ladder.py` from branch `expert-dagger`: Gumbel-MCTS self-play at 13x9 -> 15x11 -> 17x13 -> 21x15,
starting from the expert-game ladder checkpoint.

Everything lives in `My Drive/phutball/` (synced to the laptop by Google Drive for desktop):
- `data/`: `expert_data/` from the laptop (puzzle pools, held-out sets, checkpoints)
- `selfplay/`: this run: `train.log`, `log.csv`, `status.json`, checkpoints, and **`control.json`** (live settings;
  `{"reload": true}` makes the run checkpoint, pull the branch and restart)

Run all cells. If the runtime disconnects, run all again: it resumes from `selfplay/latest.pkl`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/phutball'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install mctx==0.0.6 flax optax 2>&1 | tail -1
import jax; print(jax.__version__, jax.devices())

In [ ]:
# copy the data locally (Drive reads are slow); checkpoints stay on Drive
!mkdir -p /content/data && rsync -a --exclude '*.tmp*' $DRIVE/data/expert_data /content/data/
!ls /content/data/expert_data

In [ ]:
# expert-game board ladder on the GPU, if its result isn't there yet (resumable; logs to Drive)
import os, subprocess, sys
if not os.path.exists('/content/data/expert_data/ladder/rung3_21x15.pkl'):
    LAD = f'{DRIVE}/ladder'; os.makedirs(LAD, exist_ok=True)
    cmd = (f'{sys.executable} -u -m expert.train_ladder --init /content/data/expert_data/place_run/latest.pkl '
           f'--run-dir {LAD} --data-root /content/data --batch 256 --eval-every 100')
    p = subprocess.Popen(cmd, shell=True, cwd='/content/phutball-jax', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{LAD}/stdout.log', 'a') as out:
        for line in p.stdout: print(line, end=''); out.write(line); out.flush()
    assert p.wait() == 0, 'ladder failed (see ladder/stdout.log)'
    os.makedirs('/content/data/expert_data/ladder', exist_ok=True)
    !cp {LAD}/rung*.pkl /content/data/expert_data/ladder/
print(sorted(os.listdir('/content/data/expert_data/ladder')))

In [ ]:
# initial checkpoints: the expert-game ladder result and the puzzle-trained model (the KL anchor)
import os
# self-play starts from the puzzle refresh after the ladder if it exists, else from the ladder's 21x15 result
INIT = next(p for p in ['/content/data/expert_data/refresh/latest.pkl', '/content/data/expert_data/ladder/rung3_21x15.pkl']
            if os.path.exists(p))
REF = '/content/data/expert_data/place_run/latest.pkl'
RUN = f'{DRIVE}/selfplay'
ARGS = f'--init {INIT} --puzzle-ref {REF} --run-dir {RUN} --data-root /content/data'
# A100 defaults: 256 games in parallel, 32 simulations, batch 256 (change live via control.json)
import os; os.makedirs(RUN, exist_ok=True); print(ARGS)

In [ ]:
# run loop: exit code 3 = reload requested (pull the branch and relaunch), 0 = stop, other = crash
import subprocess, sys, time
while True:
    subprocess.run('git pull -q && git log --oneline -1', shell=True, cwd='/content/phutball-jax')
    p = subprocess.Popen(f'{sys.executable} -u -m expert.selfplay_ladder {ARGS}', shell=True, cwd='/content/phutball-jax',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{RUN}/stdout.log', 'a') as out:
        for line in p.stdout:
            print(line, end=''); out.write(line); out.flush()
    code = p.wait(); print('exit code', code)
    if code == 3: continue
    if code != 0:
        print('crashed; retrying in 60 s (see stdout.log)'); time.sleep(60); continue
    break